In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("sales_data.csv", encoding='latin1')
df.head()

In [ ]:
print(df.columns)

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])

In [ ]:
df = df.sort_values(by='Order Date')

In [ ]:
df_grouped = df.groupby('Order Date')['Sales'].sum().reset_index()
df_grouped.head()

In [ ]:
df_grouped.set_index('Order Date', inplace=True)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(df_grouped.index, df_grouped['Sales'])
plt.title("Sales Over Time")
plt.show()

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(df_grouped['Sales'], order=(5,1,0))
model_fit = model.fit()

In [ ]:
forecast = model_fit.forecast(steps=10)

print(forecast)

In [ ]:
train = df_grouped['Sales'][:int(len(df_grouped)*0.8)]
test = df_grouped['Sales'][int(len(df_grouped)*0.8):]

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(train, order=(5,1,0))
model_fit = model.fit()

In [ ]:
forecast = model_fit.forecast(steps=len(test))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(test, forecast)
mse = mean_squared_error(test, forecast)
rmse = np.sqrt(mse)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(test.index, test, label='Actual')
plt.plot(test.index, forecast, label='Forecast', color='red')
plt.legend()
plt.title("Actual vs Forecast Sales")
plt.show()

In [ ]:
future_forecast = model_fit.forecast(steps=10)

print("Future Sales Prediction:")
print(future_forecast)

In [ ]:
df_lr = df_grouped.copy()

df_lr['day'] = df_lr.index.day
df_lr['month'] = df_lr.index.month
df_lr['year'] = df_lr.index.year

In [ ]:
from sklearn.model_selection import train_test_split

X = df_lr[['day', 'month', 'year']]
y = df_lr['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [ ]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

In [ ]:
lr_pred = lr_model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print("Linear Regression Results:")
print("MAE:", mean_absolute_error(y_test, lr_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))

In [ ]:
monthly_sales = df.groupby(df['Order Date'].dt.to_period('M'))['Sales'].sum()

monthly_sales.plot(figsize=(10,5), title="Monthly Sales Trend")

In [ ]:
from sklearn.model_selection import train_test_split

X = df_lr[['day', 'month', 'year']]
y = df_lr['Sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [ ]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

In [ ]:
print("\nComparison:")
print("ARIMA RMSE:", rmse)
print("Linear Regression RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(y_test.index, y_test, label='Actual')
plt.plot(y_test.index, lr_pred, label='Linear Regression')
plt.legend()
plt.title("Linear Regression vs Actual")
plt.show()